In [45]:
import teehr
import pandas as pd
# from teehr.evaluation.spark_session_utils import create_spark_session
from setup_utils import create_minio_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time
import requests
from urllib.parse import urlparse, parse_qs

from typing import Union, List, Optional

teehr.__version__

'0.6.5'

In [2]:
spark = create_minio_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using user-provided AWS credentials
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [66]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


In [31]:
def _make_request(
        endpoint: str,
        api_base_url: str,
        verify_ssl: bool = False,
        params: dict = None,
        headers: dict = None,
        timeout: int = 60
    ) -> requests.Response:
        """Make a request to the warehouse API.

        Parameters
        ----------
        endpoint : str
            API endpoint path (e.g., "collections/locations/items")
        api_base_url : str
            Base URL for the TEEHR warehouse API
        verify_ssl : bool, optional
            Whether to verify SSL certificates. Default: False
        params : dict, optional
            Query parameters for the request
        headers : dict, optional
            Request headers for the API call
        timeout : int, optional
            Request timeout in seconds. Default: 60

        Returns
        -------
        requests.Response
            Response object from the API

        Raises
        ------
        requests.HTTPError
            If the request fails
        requests.exceptions.Timeout
            If the request times out
        """
        url = endpoint if endpoint.startswith("http") else f"{api_base_url}/{endpoint}"

        try:
            response = requests.get(
                url,
                params=params or {},
                headers=headers or {},
                verify=verify_ssl,
                timeout=timeout
            )
            response.raise_for_status()
        except requests.exceptions.Timeout:
            raise

        return response

In [38]:
def _extract_next_link(payload: dict) -> Optional[str]:
        """Extract the next-page URL from a collection payload links array, if present."""
        links = payload.get("links", []) if isinstance(payload, dict) else []
        for link in links:
            if link.get("rel") == "next" and link.get("href"):
                return link["href"]
        return None

In [41]:
def _params_from_next_link(next_link: str, base_params: dict) -> tuple[str, dict]:
        """Parse a next link URL into endpoint and pagination params, preserving base filters."""
        parsed = urlparse(next_link)
        endpoint = parsed.path.lstrip("/")

        merged_params = {**base_params}
        next_params = parse_qs(parsed.query, keep_blank_values=True)
        for key in ("offset", "limit"):
            if key in next_params and next_params[key]:
                merged_params[key] = next_params[key][-1]
        return endpoint, merged_params

In [52]:
def fetch_paginated_features(
    endpoint: str,
    params: dict,
    page_size: Optional[int],
    timeout: int = 60,
) -> list:
    """Fetch all pages from a JSON items endpoint using limit/offset pagination.

    Parameters
    ----------
    endpoint : str
        API endpoint path (e.g., "collections/attributes/items")
    params : dict
        Base query parameters (without limit/offset)
    page_size : int, optional
        Number of items to request per page. If None, the API
        determines page size based on server configuration and auth.
    timeout : int, optional
        Request timeout in seconds. Default: 60

    Returns
    -------
    list
        All items accumulated across all pages
    """
    all_items = []
    base_params = {**params}
    page_params = {**base_params}
    if page_size is not None:
        page_params["limit"] = page_size
    current_offset = 0
    request_endpoint = endpoint

    while True:
        page_params['offset'] = current_offset
        
        # Temp fix for bug
        if page_params['offset'] == 70000:
            break
        
        response = _make_request(
            request_endpoint,
            "https://api.teehr.rtiamanzi.org",
            False,
            page_params,
            headers={"x-api-key": "thk_fCb09q2fWZGIpM31F2HjCerk2340ybS1kO-Zo9bERhg"},
            timeout=timeout
        )
        payload = response.json()
        links_present = isinstance(payload, dict) and "links" in payload
        page_items = payload.get("features", [])

        all_items.extend([feature["properties"] for feature in page_items])
        records_returned = payload.get("numberReturned", len(page_items))

        next_link = _extract_next_link(payload)
        if next_link:
            request_endpoint, page_params = _params_from_next_link(next_link, base_params)
            if page_size is not None:
                page_params.setdefault("limit", page_size)
            if "offset" in page_params:
                try:
                    current_offset = int(page_params["offset"])
                except (TypeError, ValueError):
                    current_offset += records_returned
            else:
                current_offset += records_returned
            continue

        if links_present:
            break

        if records_returned == 0:
            break

        current_offset += records_returned
        page_params = {**base_params}
        if page_size is not None:
            page_params["limit"] = page_size

    return all_items

In [53]:
features = fetch_paginated_features(endpoint="collections/nwmd_metrics_by_location/items",
                                    params={},
                                    page_size=10000,
                                    timeout=60
                                   )

/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.teehr.rtiamanzi.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.teehr.rtiamanzi.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.teehr.rtiamanzi.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/

In [54]:
len(features)

70000

In [63]:
df = pd.DataFrame(features)

In [ ]:
group_by = [
    "primary_location_id",
    "secondary_location_id",
    "configuration_name",
    "unit_name",
    "variable_name",
    "member",
    "quarter",
    "forecast_lead_time_bin",
    "threshold",
    "window_agg",
]

metrics = [
    s.Count(),
    s.Average(),
    s.Minimum(),
    s.Maximum(),
    dm.RelativeMean(),
    dm.RelativeMedian(),
    dm.RelativeMinimum(),
    dm.RelativeMaximum(),
    dm.RelativeStandardDeviation(),
    dm.RelativeBias(
        add_epsilon=True,
    ),
    dm.NashSutcliffeEfficiency(
        add_epsilon=True,
    ),
    dm.KlingGuptaEfficiency(
        add_epsilon=True,
    ),
    dm.PearsonCorrelation(
        add_epsilon=True,
    ),
    dm.RelativeMean(
        output_field_name="relative_mean_boot",
        bootstrap=bootstrap,
        unpack_results=True
    ),
    dm.RelativeMedian(
        output_field_name="relative_median_boot",
        bootstrap=bootstrap,
        unpack_results=True
    ),
    dm.RelativeMinimum(
        output_field_name="relative_minimum_boot",
        bootstrap=bootstrap,
        unpack_results=True
    ),
    dm.RelativeMaximum(
        output_field_name="relative_maximum_boot",
        bootstrap=bootstrap,
        unpack_results=True
    ),
    dm.RelativeStandardDeviation(
        output_field_name="relative_standard_deviation_boot",
        bootstrap=bootstrap,
        unpack_results=True
    ),
    dm.NashSutcliffeEfficiency(
        output_field_name="nash_sutcliffe_efficiency_boot",
        bootstrap=bootstrap,
        unpack_results=True
    ),
    dm.RelativeBias(
        output_field_name="relative_bias_boot",
        bootstrap=bootstrap,
        unpack_results=True
    ),
    dm.PearsonCorrelation(
        output_field_name="pearson_correlation_boot",
        bootstrap=bootstrap,
        unpack_results=True
    ),
    dm.KlingGuptaEfficiency(
        output_field_name="kling_gupta_efficiency_boot",
        bootstrap=bootstrap,
        unpack_results=True
    ),
]

In [68]:
table_name = "nwmd_metrics_by_location"

# teehr_df.write_to(table_name=table_name, write_mode="create_or_replace")

ev._load.dataframe(
    df=df,
    table_name=table_name,
    write_mode="create_or_replace",
)

metric_columns = [metric.output_field_name for metric in metrics]

properties = {
    "description": "NWM diagnostics metrics by location ID",
    "group_by": ", ".join(group_by),
    "metrics": ", ".join(metric_columns)
}

for key, value in properties.items():
    ev.spark.sql(f"""
        ALTER TABLE iceberg.teehr.{table_name} SET TBLPROPERTIES ('{key}' = '{value}')
    """)

INFO:teehr.evaluation.tables.generic_table:Getting table: nwmd_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: nwmd_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.nwmd_metrics_by_location.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.nwmd_metrics_by_location.
INFO:teehr.evaluation.write:Start writing to warehouse table 'nwmd_metrics_by_location'.
INFO:teehr.evaluation.tables.generic_table:Getting table: nwmd_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: nwmd_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.nwmd_metrics_by_location.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.nwmd_metrics_by_location.


IllegalArgumentException: Invalid last column ID: 44 < 45 (previous last column ID)

In [69]:
spark.stop()